# 실습 - 세그멘테이션

**PCB 기판의 영역 경계 검출**

**작업 개요**

- 작업 이름 — `mask-generation`
- 입력 — 이미지 한 장
- 출력 — 영역 마스크(mask) — 대상은 흰색, 나머지는 검은색
- 마스크는 이미지 형태 — 화면에 그대로 표시 가능

**앞선 작업과의 차이**

| 구분 | 분류 | 탐지 | 세그멘테이션 |
|---|---|---|---|
| 출력 | 라벨 · 점수 | 라벨 · 점수 · 상자 | **마스크** |
| 영역 표현 | 없음 | 사각형 | **실제 윤곽** |

**데이터**

- 인쇄회로기판(PCB)을 촬영한 검사용 이미지
- 부품 · 배선 · 패드가 조밀하게 배치된 구조

**모델 — SAM (Segment Anything Model)**

- 범주를 학습하지 않고 **경계 자체**를 검출
- 결과에 이름이 없음 — 무엇인지는 판별하지 않음
- 처음 보는 분야에서도 동작 — 범주에 의존하지 않기 때문

**실습 순서**

- 환경 준비
- 데이터 불러오기
- 모델 준비 · 마스크 생성
- 마스크 확인 — 하나 · 일부 · 전체

---
## 0. 환경 준비

- `datasets` — Hub 데이터셋 내려받기
- `transformers` — 사전학습 모델로 추론 수행
- `PIL` · `numpy` — 마스크 변환과 겹치기

In [ ]:
from datasets import load_dataset
from transformers import pipeline
from PIL import Image
import numpy as np
import torch, time

print("준비 완료")
print("GPU 사용 가능:", torch.cuda.is_available())

---
## 1. 데이터 준비

### 1-1. 데이터셋 불러오기

- `name="full"` — 데이터셋의 설정(config) 지정
- `split="train[:50]"` — 앞 50장만 사용
- 실제 사용은 한두 장 — 처리 시간이 길기 때문

In [ ]:
ds = load_dataset(
    "keremberke/pcb-defect-segmentation",
    name="full",
    split="train[:50]"
)

ds

### 1-2. 이미지 확인

- 부품 · 배선 · 패드가 조밀하게 배치된 구조
- 경계가 뚜렷한 요소가 많음 — 마스크가 다수 생성될 수 있음

In [ ]:
image = ds[0]["image"]

print("이미지 크기:", image.size)
print("색상 방식:", image.mode)
image

---
## 2. 모델 준비

- `task="mask-generation"` — 마스크 생성 작업
- `model` — SAM (Segment Anything Model)
- `device=0` — GPU에서 실행

SAM은 범주를 학습하지 않으므로 범주 목록이 없다.

In [ ]:
seg = pipeline(
    task="mask-generation",
    model="facebook/sam-vit-base",
    device=0
)

print("모델 준비 완료")

---
## 3. 마스크 생성

- 이미지 전체에 지점을 뿌려 **모든 경계**를 찾는 방식
- `points_per_batch` — 한 번에 처리할 지점 수
- **처리 시간이 길다** — 한 장에 수십 초 소요 가능
- 실습에서는 한두 장만 수행

In [ ]:
t0 = time.time()

out = seg(image, points_per_batch=32)

print("소요 시간:", round(time.time() - t0, 1), "초")
print("검출된 영역:", len(out["masks"]))

### 3-1. 출력 구조 확인

- `out["masks"]` — 마스크 목록
- `out["scores"]` — 각 마스크의 신뢰도
- **라벨이 없음** — 번호로만 구분

마스크의 자료형을 먼저 확인한다. PIL 이미지가 아니라 **numpy 배열**로 반환되는 경우가 있다.

In [ ]:
print("출력 항목:", list(out.keys()))
print("마스크 수:", len(out["masks"]))

m = out["masks"][0]
print("자료형:", type(m))
print("형태:", getattr(m, "shape", None))
print("값 종류:", getattr(m, "dtype", None))

### 3-2. 마스크 변환 함수

- 마스크가 numpy 배열로 오면 **PIL 이미지로 변환** 필요
- 변환하지 않으면 화면 표시·겹치기에서 오류 발생
- 불리언(True/False) · 0~1 실수 · 0~255 정수 모두 처리

In [ ]:
def to_pil_mask(mask):
    """마스크를 PIL 흑백 이미지로 변환"""
    if isinstance(mask, Image.Image):
        return mask.convert("L")

    arr = np.asarray(mask)

    if arr.dtype == bool:
        arr = arr.astype(np.uint8) * 255
    elif arr.max() <= 1:
        arr = (arr * 255).astype(np.uint8)
    else:
        arr = arr.astype(np.uint8)

    return Image.fromarray(arr, mode="L")


print("변환 함수 준비 완료")

---
## 4. 마스크 확인

### 4-1. 마스크 하나

- 대상 영역은 **흰색**, 나머지는 **검은색**
- 번호를 바꾸어 다른 영역 확인

In [ ]:
to_pil_mask(out["masks"][0])

### 4-2. 원본에 겹치기

- `convert("RGBA")` — 투명도 처리가 가능한 형식으로 변환
- `Image.new` — 반투명 색 판 생성
- `paste(..., mask)` — 마스크 영역에만 색 입히기

In [ ]:
def overlay_mask(img, mask, color=(255, 0, 0, 110)):
    base = img.convert("RGBA")
    layer = Image.new("RGBA", img.size, color)
    base.paste(layer, (0, 0), to_pil_mask(mask))
    return base


overlay_mask(image, out["masks"][0])

### 4-3. 전체 마스크 색칠

- 검출된 **모든 영역**을 서로 다른 색으로 표시
- `seed` 고정 — 실행할 때마다 같은 색 재현
- **넓은 마스크부터** 그리기 — 작은 영역이 위에 남아 잘 보임

In [ ]:
def overlay_all(img, masks, alpha=110, seed=0):
    """모든 마스크를 서로 다른 색으로 겹쳐 표시"""
    rng = np.random.default_rng(seed)
    base = img.convert("RGBA")

    # 넓은 마스크부터 그리기
    order = sorted(
        range(len(masks)),
        key=lambda i: np.asarray(masks[i]).sum(),
        reverse=True,
    )

    for i in order:
        color = tuple(int(c) for c in rng.integers(60, 255, 3)) + (alpha,)
        layer = Image.new("RGBA", img.size, color)
        base.paste(layer, (0, 0), to_pil_mask(masks[i]))

    return base


print("전체 마스크:", len(out["masks"]), "개")
overlay_all(image, out["masks"])

### 4-4. 마스크가 많을 때

- 영역이 수백 개면 화면이 뭉개짐
- **상위 일부만** 표시하거나 **작은 영역을 제외**

In [ ]:
# 상위 30개만
overlay_all(image, out["masks"][:30])

In [ ]:
# 넓이가 일정 이상인 마스크만
big = [m for m in out["masks"] if np.asarray(m).sum() > 500]

print(f"{len(out['masks'])}개 → {len(big)}개")
overlay_all(image, big)

**확인 사항**

- 부품 · 패턴 단위로 경계가 나뉨
- 각 영역에 **이름이 없음** — 무엇인지는 알 수 없음
- 결함 여부 판별은 불가 — 경계만 검출

SAM은 시각적으로 뚜렷한 경계를 찾는 모델이다. 결함은 정상 패턴과 시각적으로 미묘하게 다를 뿐이므로, SAM이 구분할 근거가 없다.

---
## 5. 다른 기판 확인

- 이미지 번호를 바꾸어 다른 기판 확인
- `points_per_batch` 를 조절하여 처리 시간 변화 관찰
- 본인 분야 이미지에 적용 — 경계가 뚜렷한 대상일수록 잘 동작

In [ ]:
n = 1
img2 = ds[n]["image"]

t0 = time.time()
out2 = seg(img2, points_per_batch=32)
print("소요:", round(time.time() - t0, 1), "초 / 영역:", len(out2["masks"]))

overlay_all(img2, out2["masks"])

---
## 6. 정리

**수행 내용**

- 범주를 학습하지 않은 모델(SAM)로 경계 검출
- 이미지 한 장 → 영역 마스크 다수 생성
- 마스크를 원본에 겹쳐 시각적으로 확인 — 하나 · 일부 · 전체

**앞선 작업과의 차이**

- 분류 — 이미지 전체에 라벨 1개
- 탐지 — 대상마다 사각형 위치
- 세그멘테이션 — 대상의 **실제 윤곽**

**SAM의 특징과 한계**

- 처음 보는 분야에서도 동작 — 범주에 의존하지 않음
- 결과에 **이름이 없음** — 무엇인지는 판별하지 않음
- 결함 여부 등 판별이 필요하면 **별도 분류 단계** 필요
- 처리 시간이 길어 대량 처리에는 부적합

**실습에서 만난 문제**

- 마스크가 numpy 배열로 반환 — PIL 이미지로 변환 후 사용
- 출력 형태는 모델·버전에 따라 상이 — 자료형 확인이 선행되어야 함

**연결**

- 도입부 의료 연구 사례(H-SAM) — SAM에 디코더를 추가하여 의료영상에 적응
- 범주 판별이 필요한 경우 → 후반부 미세조정

---

데이터 출처: keremberke/pcb-defect-segmentation (Hugging Face Hub 공개)